# Inside the Tokenizer

Starter notebook. Fill in the TODOs and **run every cell** before committing — the graders need to see your outputs.


In [1]:
# Setup
#!pip install -r requirements.txt
import tiktoken
import transformers  # the Hugging Face Transformers library
from transformers import AutoTokenizer  # a second tokenizer (no model weights needed)
import pandas as pd

## Task 1 — Tokenize and compare

Pick a paragraph of mixed content (English + code + another language + an emoji), tokenize with **two** tokenizers, and show tokens, IDs, and counts side by side.


In [2]:
TEXT = """I showed my masterpiece to the grown-ups, and asked them whether the drawing frightened them.
But they answered: "Frighten? Why should any one be frightened by a hat?"
My drawing was not a picture of a hat. It was a picture of a boa constrictor digesting an elephant. But since
the grown-ups were not able to understand it, I made another drawing: I drew the inside of a boa
constrictor, so that the grown-ups could see it clearly. They always need to have things explained. My
Drawing Number Two looked like this. 
The grown-ups' response, this time, was to advise me to lay aside my drawings of boa constrictors, whether
from the inside or the outside, and devote myself instead to geography, history, arithmetic, and grammar.
That is why, at the age of six, I gave up what might have been a magnificent career as a painter. I had been
disheartened by the failure of my Drawing Number One and my Drawing Number Two. Grown-ups never
5
understand anything by themselves, and it is tiresome for children to be always and forever explaining
things to them.
So then I chose another profession, and learned to pilot airplanes. I have flown a little over all parts of the
world; and it is true that geography has been very useful to me. At a glance I can distinguish China from
Arizona. If one gets lost in the night, such knowledge is valuable.
In the course of this life I have had a great many encounters with a great many people who have been
concerned with matters of consequence. I have lived a great deal among grown-ups. I have seen them
intimately, close at hand. And that hasn't much improved my opinion of them.
Whenever I met one of them who seemed to me at all clear-sighted, I tried the experiment of showing him
my Drawing Number One, which I have always kept. I would try to find out, so, if this was a person of true
understanding. But, whoever it was, he, or she, would always say:
"That is a hat."
Then I would never talk to that person about boa constrictors, or primeval forests, or stars. I would bring
myself down to his level. I would talk to him about bridge, and golf, and politics, and neckties. And the
grown-up would be greatly pleased to have met such a sensible man"""

enc = tiktoken.get_encoding("cl100k_base")
ids = enc.encode(TEXT)
tokens = [enc.decode([i]) for i in ids]
print("count:", len(ids))
print("tokens:", tokens[:30])
print("ids:", ids[:30])


tokenizer = AutoTokenizer.from_pretrained("gpt2")
inputs = tokenizer(TEXT, return_tensors="pt")
print("count:", len(inputs["input_ids"][0]))
print("tokens:", tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][:30]))
print("ids:", inputs["input_ids"][0][:30])


df = pd.DataFrame(
    {"tiktoken": tokens[:30],
        "tiktoken_id": ids[:30],
        "transformers": tokenizer.convert_ids_to_tokens(inputs["input_ids"][0][:30]),
        "transformers_id": inputs["input_ids"][0][:30].tolist()})
# TODO: do the same with a second tokenizer (e.g. AutoTokenizer.from_pretrained(...))
# TODO: print a small side-by-side count table
df

count: 511
tokens: ['I', ' showed', ' my', ' masterpiece', ' to', ' the', ' grown', '-ups', ',', ' and', ' asked', ' them', ' whether', ' the', ' drawing', ' frightened', ' them', '.\n', 'But', ' they', ' answered', ':', ' "', 'F', 'right', 'en', '?', ' Why', ' should', ' any']
ids: [40, 8710, 856, 59831, 311, 279, 15042, 27859, 11, 323, 4691, 1124, 3508, 279, 13633, 60178, 1124, 627, 4071, 814, 19089, 25, 330, 37, 1315, 268, 30, 8595, 1288, 904]
count: 532
tokens: ['I', 'Ġshowed', 'Ġmy', 'Ġmasterpiece', 'Ġto', 'Ġthe', 'Ġgrown', '-', 'ups', ',', 'Ġand', 'Ġasked', 'Ġthem', 'Ġwhether', 'Ġthe', 'Ġdrawing', 'Ġfrightened', 'Ġthem', '.', 'Ċ', 'But', 'Ġthey', 'Ġanswered', ':', 'Ġ"', 'F', 'right', 'en', '?', 'ĠWhy']
ids: tensor([   40,  3751,   616, 30669,   284,   262,  7334,    12,  4739,    11,
          290,  1965,   606,  1771,   262,  8263, 24776,   606,    13,   198,
         1537,   484,  9373,    25,   366,    37,  3506,   268,    30,  4162])


,tiktoken,tiktoken_id,transformers,transformers_id
0,I,40,I,40
1,showed,8710,Ġshowed,3751
2,my,856,Ġmy,616
3,masterpiece,59831,Ġmasterpiece,30669
4,to,311,Ġto,284
5,the,279,Ġthe,262
6,grown,15042,Ġgrown,7334
7,-ups,27859,-,12
8,",",11,ups,4739
9,and,323,",",11


**Where do the two tokenizers disagree most, and why?**

> TODO


## Task 2 — Hunt the weird cases

Investigate **at least three**: rare vs common word, English vs another language, code vs prose, leading-space, emoji/accents. Show the token counts that prove your point.


In [3]:
cases = {
    "common_word": "running",
    "rare_word": "supercalifragilisticexpialidocious",
    "emoji": "😀",
    "non_latin": "你好",
}

for name, s in cases.items():
    print(f"{name:15} {len(enc.encode(s)):3}  {s!r}")


common_word       1  'running'
rare_word        11  'supercalifragilisticexpialidocious'
emoji             2  '😀'
non_latin         2  '你好'


**Observations (one sentence per case):**

For common word it used only 1 token, but for the rare word, although it is only 3-4 times longer than the common word, it used 11 tokens. On the other hand for emoji(it is only 1 character) it used 2 tokens. for non-latin case again although it is only a word, there used 2 tokens for 1 character


## Task 3 — Token + cost estimator


In [4]:
def estimate(text, price_in_per_1k, price_out_per_1k, expected_output_tokens):
    """Return (input_token_count, projected_total_cost)."""
    n_in = len(enc.encode(text))
    cost = (n_in / 1000) * price_in_per_1k + (expected_output_tokens / 1000) * price_out_per_1k
    return n_in, cost

short = "How do I reset my password in arch?"
long = """I showed my masterpiece to the grown-ups, and asked them whether the drawing frightened them.
But they answered: "Frighten? Why should any one be frightened by a hat?"
My drawing was not a picture of a hat. It was a picture of a boa constrictor digesting an elephant. But since
the grown-ups were not able to understand it, I made another drawing: I drew the inside of a boa
constrictor, so that the grown-ups could see it clearly. They always need to have things explained. My
Drawing Number Two looked like this. 
The grown-ups' response, this time, was to advise me to lay aside my drawings of boa constrictors, whether
from the inside or the outside, and devote myself instead to geography, history, arithmetic, and grammar.
That is why, at the age of six, I gave up what might have been a magnificent career as a painter. I had been
disheartened by the failure of my Drawing Number One and my Drawing Number Two. Grown-ups never
5
understand anything by themselves, and it is tiresome for children to be always and forever explaining
things to them.
So then I chose another profession, and learned to pilot airplanes. I have flown a little over all parts of the
world; and it is true that geography has been very useful to me. At a glance I can distinguish China from
Arizona. If one gets lost in the night, such knowledge is valuable.
In the course of this life I have had a great many encounters with a great many people who have been
concerned with matters of consequence. I have lived a great deal among grown-ups. I have seen them
intimately, close at hand. And that hasn't much improved my opinion of them.
Whenever I met one of them who seemed to me at all clear-sighted, I tried the experiment of showing him
my Drawing Number One, which I have always kept. I would try to find out, so, if this was a person of true
understanding. But, whoever it was, he, or she, would always say:
"That is a hat."
Then I would never talk to that person about boa constrictors, or primeval forests, or stars. I would bring
myself down to his level. I would talk to him about bridge, and golf, and politics, and neckties. And the
grown-up would be greatly pleased to have met such a sensible man"""
# TODO: run on three different inputs (short question, long document, code file)
# with a small made-up price table, and print the results.
print(estimate(short, 0.075, 0.30, 50))
print(estimate(long, 0.075, 0.30, 50))

(9, 0.015674999999999998)
(511, 0.053325)


**Which input is most expensive, and is it what you expected?**

The long document is the most expensive input by far, and yes, that is what I expected because it has many more input tokens, which dominates the total estimated cost.